# CORAL ST ingest — tutorial test notebook

Ingests every sample in `STFM/Data/CORAL_st_tutorial_data/` and verifies each
store against the CORAL contract (frozen core: root image pyramid +
`st/<tech>/` AnnData). Covers the natively-ported imaging-ST readers
**CosMx** and **G4X** plus **Visium, Visium HD, Xenium, HEST, legacy ST**.

Self-contained: it unpacks the archive-staged platforms into `_prepared/`,
then auto-detects and ingests each sample.

**Kernel:** the CORAL fork's virtualenv (`Code/CORAL/.venv`). If import fails:
`uv run --project Code/CORAL python -m ipykernel install --user --name coral`.

> Two folders are intentionally **excluded**: `atera_wta_breast_cancer_500um`
> (a separate upcoming IO task) and the bare `xenium/` (only an alignment CSV,
> no bundle). Xenium is covered by the small `xenium_prime5k_lung_cancer_500um`.

In [ ]:
import os, sys, json, time, tarfile, zipfile
os.environ.setdefault("MPLBACKEND", "Agg")
from pathlib import Path
import numpy as np, pandas as pd

CODE_DIR = Path("/mnt/beegfs/storage/Yuzhou_tmp/Projects/CORAL/Code/CORAL")
DATA     = Path("/mnt/beegfs/storage/Yuzhou_tmp/Projects/STFM/Data/CORAL_st_tutorial_data")
HERE     = CODE_DIR / "tutorials" / "_st_ingest_work"
PREP     = HERE / "_prepared"
OUT_DIR  = HERE / "_stores"

OUT_DIR.mkdir(parents=True, exist_ok=True); PREP.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE_DIR / "src"))
import coral
from coral.st import ST_AVAILABLE
print("coral", coral.__version__, "| ST_AVAILABLE:", ST_AVAILABLE, "| python", sys.version.split()[0])
assert ST_AVAILABLE, "install the st extra:  uv sync --extra st"
print("datasets in folder:", [p.name for p in sorted(DATA.iterdir()) if p.is_dir()])

## Prepare bundles — unpack the archive-staged platforms

CosMx, G4X and the small Xenium are already complete bundles in the data
folder. Visium / Visium HD / HEST / legacy ST are staged under `<platform>/raw/`
(some as archives); this cell arranges each into the layout its reader expects.
Idempotent — safe to re-run.

In [ ]:
def _link(src: Path, dst_dir: Path):
    dst_dir.mkdir(parents=True, exist_ok=True)
    d = dst_dir / src.name
    if not d.exists(): d.symlink_to(src)

# Visium: counts h5 + full-res image + unpacked spatial/
v = PREP / "visium"
if not (v / "spatial").is_dir():
    for f in (DATA / "visium" / "raw").glob("*"):
        if f.suffix in (".h5", ".tif"): _link(f, v)
    with tarfile.open(DATA / "visium" / "raw" / "V1_Human_Lymph_Node_spatial.tar.gz") as t:
        t.extractall(v)

# Visium HD: unzip the outs bundle at the root (binned_outputs/ must be top-level)
vhd = PREP / "visium-hd"
if not (vhd / "binned_outputs").is_dir():
    with zipfile.ZipFile(DATA / "visium-hd" / "raw" / "Visium_HD_Tiny_3prime_Dataset_outs.zip") as z:
        z.extractall(vhd)

# HEST + legacy ST: symlink the raw files
for f in (DATA / "hest" / "raw").glob("*"): _link(f, PREP / "hest")
for f in (DATA / "spatial-transcriptomics" / "raw").glob("*"): _link(f, PREP / "legacy-st")

print("prepared:", sorted(p.name for p in PREP.iterdir() if p.is_dir()))

## Helpers — ingest one sample (auto-detect), then verify its store

In [ ]:
from coral.st.pipeline import ingest_one
from coral.st.detect import detect_technology
import zarr, anndata as ad

def verify_store(store: Path) -> dict:
    r = zarr.open_group(str(store), mode="r"); attrs = dict(r.attrs)
    rec = next(p.name for p in (store / "st").iterdir() if (p / "config.json").is_file())
    a = ad.read_zarr(str(store / "st" / rec))
    cfg = json.loads((store / "st" / rec / "config.json").read_text())
    X = a.X; data = X.data if hasattr(X, "data") else np.asarray(X)
    sp = a.obsm.get("spatial")
    c, h, w = (int(x) for x in r["0"].shape)
    frac_in = float((((sp[:, 0] >= 0) & (sp[:, 0] <= w) & (sp[:, 1] >= 0)
                      & (sp[:, 1] <= h)).mean())) if sp is not None else 0.0
    return {"store": store.stem, "technology": attrs.get("technology"),
            "image_cyx": (c, h, w), "levels": len(list(r.array_keys())),
            "nuclear": attrs.get("nuclear_channel"), "mpp": attrs.get("mpp"),
            "n_obs": int(a.n_obs), "n_vars": int(a.n_vars),
            "raw_int_counts": bool(np.allclose(data, np.round(data))),
            "coords_in_img": round(frac_in, 3), "obs_unit": cfg.get("obs_unit"),
            "frozen_core_ok": ("0" in list(r.array_keys()))
                              and (store / "state.json").is_file()
                              and (store / "OME" / "METADATA.ome.xml").is_file()}

def run_ingest(sample_dir: Path) -> dict:
    t0 = time.time()
    store = ingest_one(sample_dir, OUT_DIR)      # technology auto-detected
    res = verify_store(Path(store))
    res["seconds"] = round(time.time() - t0, 1); res["ok"] = True; res["note"] = ""
    return res

## Ingest every sample in the tutorial folder

In [ ]:
SAMPLES = [
    DATA / "cosmx_INDEPTH_DFCI_2fov",
    DATA / "g4x_tonsil_rep1_subsample",
    PREP / "visium",
    PREP / "visium-hd",
    PREP / "hest",
    PREP / "legacy-st",
    DATA / "xenium_prime5k_lung_cancer_500um",   # small Xenium, complete bundle
]
results = []
for d in SAMPLES:
    try:
        tech = detect_technology(d)
    except Exception as e:
        results.append({"store": d.name, "technology": None, "ok": False,
                        "note": f"not detected ({type(e).__name__})"}); continue
    print(f"ingesting {d.name} ({tech}) ...", flush=True)
    try:
        results.append(run_ingest(d))
    except Exception as e:
        results.append({"store": d.name, "technology": tech, "ok": False,
                        "note": f"{type(e).__name__}: {e}"[:110]})
print("done")

## Summary

In [ ]:
cols = ["store", "technology", "ok", "n_obs", "n_vars", "image_cyx", "levels",
        "nuclear", "mpp", "raw_int_counts", "coords_in_img", "frozen_core_ok",
        "obs_unit", "seconds", "note"]
df = pd.DataFrame(results).reindex(columns=cols)
print(f"{int(df['ok'].fillna(False).sum())}/{len(df)} ingested OK")
df

## QC — nuclear (or first) channel per store, should show tissue not black

In [ ]:
import matplotlib.pyplot as plt
ok = [r for r in results if r.get("ok")]
n = len(ok); cols_ = min(4, n); rows_ = (n + cols_ - 1) // cols_
fig, axes = plt.subplots(rows_, cols_, figsize=(4.5 * cols_, 4.5 * rows_))
axes = np.atleast_1d(axes).ravel()
for ax, r in zip(axes, ok):
    rr = zarr.open_group(str(OUT_DIR / f"{r['store']}.zarr"), mode="r")
    lvl = next((L for L in ("4", "3", "2", "1") if L in list(rr.array_keys())), "0")
    chans = dict(rr.attrs)["channels"]; nuc = rr.attrs.get("nuclear_channel")
    ci = next((i for i, c in enumerate(chans) if c["marker"] == nuc), 0)
    img = np.asarray(rr[lvl][ci]).astype(float)
    ax.imshow(np.clip(img / (np.percentile(img, 99.5) + 1e-6), 0, 1), cmap="gray")
    ax.set_title(f"{r['technology']} — {r['store']}\nch={chans[ci]['marker']}", fontsize=9)
    ax.axis("off")
for ax in axes[len(ok):]: ax.axis("off")
plt.tight_layout()
qc = HERE / "_st_ingest_qc.png"; plt.savefig(qc, dpi=80, bbox_inches="tight")
print("saved:", qc); plt.show()